# **Indexing, Selection, and Summary Operations**

**DS112 — Computer Programming**

**Week 14:** · **Suggested class time: 5 hours (2-hour lecture + 3-hour laboratory)**

**ILO :** Perform selection, filtering, missing-value handling, and summary operations using pandas.

Most analysis questions are about **subsets** of the data: *Which students failed the midterm? What is the average final of section DS1A? Who has missing scores?* This lesson covers how to **select**, **filter**, **clean**, and **summarize** data in pandas.

In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

grades = pd.read_csv("data/grades.csv", index_col="student_id")
grades.head()

## **Three Ways to Select**

| Syntax | Selects by | Example |
| ------ | ---------- | ------- |
| `df["col"]`, `df[["a", "b"]]` | column name(s) | `grades["final"]` |
| `df.loc[rows, cols]` | **labels** (index values, column names) | `grades.loc["2026-0003", "final"]` |
| `df.iloc[rows, cols]` | **integer positions** | `grades.iloc[2, 5]` |

> **Analogy.** `loc` is like finding a seat by its **ticket label** (*Row F, Seat 12*); `iloc` is like counting **positions** (*the third row, the sixth seat*).

In [ ]:
print(grades["final"].head(3))
print(grades.loc["2026-0003", "final"])
print(grades.iloc[2, 5])
grades.loc["2026-0001":"2026-0004", ["name", "quiz", "final"]]    # label slices INCLUDE the end

In [ ]:
grades.iloc[:3, 1:4]                                                # position slices exclude the end

## **Filtering with Boolean Conditions**

In [ ]:
failed_midterm = grades[grades["midterm"] < 75]
print(len(failed_midterm), "students below 75 on the midterm")

ds1a_high = grades[(grades["section"] == "DS1A") & (grades["final"] >= 85)]
ds1a_high[["name", "section", "final"]]

Combine conditions with `&` (and), `|` (or), `~` (not), and wrap each condition in **parentheses**.

Other handy filters:

In [ ]:
print(grades[grades["section"].isin(["DS1B", "DS1D"])].shape)
print(grades[grades["quiz"].between(70, 80)]["name"].tolist())
grades.query("final >= 90 and lab >= 85")[["name", "lab", "final"]]

## **Changing Values Safely**

Use `loc` with a condition to change only the matching rows — in one step:

In [ ]:
adjusted = grades.copy()
adjusted.loc[adjusted["final"] < 60, "final"] = 60           # floor very low finals at 60 (an example policy)
print((grades["final"] < 60).sum(), "values changed;", adjusted["final"].min(), "is the new minimum")

> Avoid "chained" assignments like `df[df["x"] > 0]["y"] = 1` — they modify a temporary copy, not your DataFrame. Use `df.loc[condition, "y"] = 1`.

## **Missing Values**

pandas marks missing data as `NaN`. The main tools:

| Method | Purpose |
| ------ | ------- |
| `isna()`, `notna()` | find missing values |
| `dropna()` | remove rows (or columns) with missing values |
| `fillna(value)` | replace missing values |

In [ ]:
print(grades.isna().sum())
grades[grades.isna().any(axis=1)][["name", "lab", "midterm"]]

In [ ]:
complete = grades.dropna()
filled = grades.fillna({"midterm": grades["midterm"].mean().round(1), "lab": 0})
print(len(grades), "→", len(complete), "rows after dropna")
print(filled.loc[grades["midterm"].isna(), ["name", "midterm"]])

Choosing **how** to handle missing data is a decision with consequences: dropping rows loses students; filling with the mean may hide a real absence. Always state the choice you made.

## **Summary Operations**

In [ ]:
scores = grades[["quiz", "lab", "midterm", "final"]]

print(scores.mean().round(2))                   # per column
print(scores.mean(axis=1).head().round(2))      # per student
print(grades["section"].value_counts())
print(grades["section"].nunique(), grades["section"].unique())
print(grades.loc[grades["final"].idxmax(), "name"], "has the highest final")

In [ ]:
grades.nlargest(3, "final")[["name", "section", "final"]]

In [ ]:
grades[["quiz", "final"]].agg(["mean", "median", "std", "min", "max"]).round(2)

## **Method Chaining**

Several steps can be written as one readable chain:

In [ ]:
(
    grades
    .dropna()
    .assign(average=lambda d: d[["quiz", "lab", "midterm", "final"]].mean(axis=1))
    .query("average >= 85")
    .sort_values("average", ascending=False)
    [["name", "section", "average"]]
    .round(2)
)

:::{admonition} Paradigm lens · declarative
:class: tip

`grades.query("average >= 85")` describes **which rows you want** rather than how to loop through them — much like SQL's `WHERE` clause. Combined with method chaining, pandas code reads as a description of the result.
:::

## **Summary**

| Task | Code |
| ---- | ---- |
| select columns | `df["a"]`, `df[["a", "b"]]` |
| select by label / position | `df.loc[r, c]` / `df.iloc[r, c]` |
| filter rows | `df[cond]`, `isin`, `between`, `query` |
| set values | `df.loc[cond, "col"] = value` |
| missing values | `isna`, `dropna`, `fillna` |
| summaries | `mean`, `agg`, `value_counts`, `nunique`, `idxmax`, `nlargest` |

## **References**

Maulion, M. R. (2022). *DS 112 computer programming* [Instructional material]. University of Science and Technology of Southern Philippines.

McKinney, W. (2022). *Python for data analysis: Data wrangling with pandas, NumPy, and Jupyter* (3rd ed.). O'Reilly Media. https://wesmckinney.com/book/

The pandas development team. (2026). *Indexing and selecting data*. https://pandas.pydata.org/docs/user_guide/indexing.html

The pandas development team. (2026). *pandas documentation*. https://pandas.pydata.org/docs/

VanderPlas, J. (2023). *Python data science handbook: Essential tools for working with data* (2nd ed.). O'Reilly Media. https://jakevdp.github.io/PythonDataScienceHandbook/